# All Embeddings Neural Network Optimization

This notebook performs a comprehensive evaluation of Neural Networks across all available embedding datasets.

**Objectives:**
1.  **Global Comparison**: Evaluate all embeddings (CodeBERT, Gemini, etc.) with different Sampling Strategies (Original, SMOTE, UnderSample) and NN Architectures.
2.  **Feature Engineering**: Optimize the best combination using Dimensionality Reduction (PCA, SelectKBest, UMAP).
3.  **Final Pipeline**: Build and save the best performing model pipeline.

In [11]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
import warnings
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split
import umap

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Suppress TF logs

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


In [3]:
# Helper Functions

def load_data(file_path):
    """Loads CSV, separates features and target, encodes target."""
    df = pd.read_csv(file_path)
    X = df.drop(columns=['label'])
    y = df['label']
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    return X, y_encoded, le

def create_nn_model(input_dim, num_classes, structure='simple'):
    """Creates a Keras model based on the specified structure."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    if structure == 'simple':
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
        
    elif structure == 'deep':
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
        
    elif structure == 'wide':
        model.add(layers.Dense(256, activation='relu'))
        model.add(layers.Dropout(0.4))
        
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def get_embedding_files(directory='embeddings'):
    """Returns a list of CSV files in the directory."""
    return glob.glob(os.path.join(directory, '*_embeddings.csv'))

print("Helper functions defined.")

Helper functions defined.


In [4]:
# Experiment 1: Global Comparison (Embeddings x Sampling x Structure)

embedding_files = get_embedding_files()
samplers = {
    'Original': None,
    'SMOTE': SMOTE(random_state=42),
    'UnderSample': RandomUnderSampler(random_state=42)
}
structures = ['simple', 'deep', 'wide']

results_exp1 = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Found {len(embedding_files)} embedding files.")

for file_path in embedding_files:
    emb_name = os.path.basename(file_path).replace('_embeddings.csv', '')
    print(f"\nProcessing {emb_name}...")
    
    try:
        X, y, le = load_data(file_path)
        input_dim = X.shape[1]
        num_classes = len(np.unique(y))
        
        for sampler_name, sampler in samplers.items():
            for structure in structures:
                print(f"  - {sampler_name} | {structure}...", end="")
                
                # Wrap Keras model for Scikit-Learn
                # We need to pass arguments to the build_fn
                # Note: scikeras uses model=... instead of build_fn=... in newer versions, 
                # but KerasClassifier from tensorflow.keras.wrappers (if used) uses build_fn.
                # We are using scikeras.wrappers.KerasClassifier here.
                
                def model_builder():
                    return create_nn_model(input_dim, num_classes, structure)

                clf = KerasClassifier(model=model_builder, epochs=50, batch_size=32, verbose=0)
                
                steps = [('scaler', StandardScaler())]
                if sampler:
                    steps.append(('sampler', sampler))
                steps.append(('model', clf))
                
                pipeline = ImbPipeline(steps)
                
                scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=1) # n_jobs=1 for Keras safety
                avg_score = np.mean(scores)
                
                results_exp1.append({
                    'Embedding': emb_name,
                    'Sampling': sampler_name,
                    'Structure': structure,
                    'Accuracy': avg_score
                })
                print(f" {avg_score:.4f}")
                
    except Exception as e:
        print(f"Error processing {emb_name}: {e}")

df_results_exp1 = pd.DataFrame(results_exp1)
display(df_results_exp1.sort_values(by='Accuracy', ascending=False))

# Select Best Combination
best_row = df_results_exp1.loc[df_results_exp1['Accuracy'].idxmax()]
best_embedding = best_row['Embedding']
best_sampling = best_row['Sampling']
best_structure = best_row['Structure']

print(f"\nBest Combination: {best_embedding} + {best_sampling} + {best_structure} (Acc: {best_row['Accuracy']:.4f})")

Found 6 embedding files.

Processing RoBERTa...
  - Original | simple...  - Original | simple...

2025-12-04 11:40:43.965823: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


 0.6469
  - Original | deep... 0.6392
  - Original | wide... 0.6392
  - Original | wide... 0.6462
  - SMOTE | simple... 0.6462
  - SMOTE | simple... 0.6301
  - SMOTE | deep... 0.6301
  - SMOTE | deep... 0.6483
  - SMOTE | wide... 0.6483
  - SMOTE | wide... 0.6545
  - UnderSample | simple... 0.6545
  - UnderSample | simple... 0.5944
  - UnderSample | deep... 0.5944
  - UnderSample | deep... 0.5979
  - UnderSample | wide... 0.5979
  - UnderSample | wide... 0.6147

Processing Jina-V2...
 0.6147

Processing Jina-V2...
  - Original | simple...  - Original | simple... 0.6860
  - Original | deep... 0.6860
  - Original | deep... 0.6944
  - Original | wide... 0.6944
  - Original | wide... 0.6881
  - SMOTE | simple... 0.6881
  - SMOTE | simple... 0.6727
  - SMOTE | deep... 0.6727
  - SMOTE | deep... 0.6776
  - SMOTE | wide... 0.6776
  - SMOTE | wide... 0.6902
  - UnderSample | simple... 0.6902
  - UnderSample | simple... 0.6371
  - UnderSample | deep... 0.6371
  - UnderSample | deep... 0.6378
  

,Embedding,Sampling,Structure,Accuracy
20,Gemini,Original,wide,0.811189
23,Gemini,SMOTE,wide,0.804895
21,Gemini,SMOTE,simple,0.803497
22,Gemini,SMOTE,deep,0.798601
18,Gemini,Original,simple,0.797203
19,Gemini,Original,deep,0.794406
26,Gemini,UnderSample,wide,0.781818
38,Nomic,Original,wide,0.774825
37,Nomic,Original,deep,0.771329
36,Nomic,Original,simple,0.766434



Best Combination: Gemini + Original + wide (Acc: 0.8112)


In [6]:
# Experiment 2: Feature Engineering Optimization

print(f"\nRunning Experiment 2 on Best Combination: {best_embedding}...")

# Load Best Data
best_file_path = os.path.join('embeddings', f"{best_embedding}_embeddings.csv")
X_best, y_best, le_best = load_data(best_file_path)
num_classes_best = len(np.unique(y_best))

# Get Best Sampler
best_sampler_obj = samplers[best_sampling]

# Define Reductions
reductions = {
    'Original': None,
    'PCA': PCA(n_components=0.95),
    'SelectKBest': SelectKBest(f_classif, k=300),
    'UMAP': umap.UMAP(n_components=50, random_state=42)
}

# Dynamic NN builder that infers input dimension from pipeline metadata
def model_builder_best(meta):
    input_dim = meta["n_features_in_"]
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    if best_structure == 'simple':
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
    elif best_structure == 'deep':
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
    elif best_structure == 'wide':
        model.add(layers.Dense(256, activation='relu'))
        model.add(layers.Dropout(0.4))
    model.add(layers.Dense(num_classes_best, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

results_exp2 = []

for red_name, reducer in reductions.items():
    print(f"Evaluating {red_name}...", end="")
    clf = KerasClassifier(model=model_builder_best, epochs=50, batch_size=32, verbose=0)
    
    steps = [('scaler', StandardScaler())]
    if best_sampler_obj:
        steps.append(('sampler', best_sampler_obj))
    if reducer:
        steps.append(('reducer', reducer))
    steps.append(('model', clf))
    
    pipeline = ImbPipeline(steps)
    
    try:
        scores = cross_val_score(pipeline, X_best, y_best, cv=cv, scoring='accuracy', n_jobs=1)
        avg_score = np.mean(scores)
        results_exp2.append({'Reduction': red_name, 'Accuracy': avg_score})
        print(f" {avg_score:.4f}")
    except Exception as e:
        print(f" Failed: {e}")

if not results_exp2:
    raise RuntimeError("Experiment 2 failed for all feature engineering strategies. Check logs above for details.")

df_results_exp2 = pd.DataFrame(results_exp2)
display(df_results_exp2.sort_values(by='Accuracy', ascending=False))

best_reduction_row = df_results_exp2.loc[df_results_exp2['Accuracy'].idxmax()]
best_reduction = best_reduction_row['Reduction']
print(f"\nBest Reduction: {best_reduction}")


Running Experiment 2 on Best Combination: Gemini...
Evaluating Original...Evaluating Original... 0.8056
Evaluating PCA... 0.8007
Evaluating SelectKBest... 0.7944
Evaluating UMAP... 0.5825


,Reduction,Accuracy
0,Original,0.805594
1,PCA,0.800699
2,SelectKBest,0.794406
3,UMAP,0.582517



Best Reduction: Original


In [8]:
# Experiment 3: Hyperparameter Tuning

print(f"\nRunning Hyperparameter Tuning on Best Combination: {best_embedding} + {best_sampling} + {best_reduction}...")

best_reducer_obj = reductions[best_reduction]

# Define Model Builder for Tuning that adapts to feature count
def tuning_model_builder(meta, structure='simple', learning_rate=0.001):
    input_dim = meta["n_features_in_"]
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    if structure == 'simple':
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
    elif structure == 'deep':
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(64, activation='relu'))
        model.add(layers.Dropout(0.3))
        model.add(layers.Dense(32, activation='relu'))
    elif structure == 'wide':
        model.add(layers.Dense(256, activation='relu'))
        model.add(layers.Dropout(0.4))
    model.add(layers.Dense(num_classes_best, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

clf_tune = KerasClassifier(model=tuning_model_builder, verbose=0)

# Parameter Grid
param_grid = {
    'model__model__structure': ['simple', 'deep', 'wide'],
    'model__model__learning_rate': [0.001, 0.01],
    'model__epochs': [50, 100],
    'model__batch_size': [32, 64]
}

steps = [('scaler', StandardScaler())]
if best_sampler_obj:
    steps.append(('sampler', best_sampler_obj))
if best_reducer_obj:
    steps.append(('reducer', best_reducer_obj))
steps.append(('model', clf_tune))

pipeline_tune = ImbPipeline(steps)

# Grid Search
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(pipeline_tune, param_grid, cv=3, scoring='accuracy', n_jobs=1, verbose=1)
grid_search.fit(X_best, y_best)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Accuracy: {grid_search.best_score_:.4f}")

best_pipeline_final = grid_search.best_estimator_


Running Hyperparameter Tuning on Best Combination: Gemini + Original + Original...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best Parameters: {'model__batch_size': 32, 'model__epochs': 50, 'model__model__learning_rate': 0.001, 'model__model__structure': 'deep'}
Best Accuracy: 0.1118
Best Parameters: {'model__batch_size': 32, 'model__epochs': 50, 'model__model__learning_rate': 0.001, 'model__model__structure': 'deep'}
Best Accuracy: 0.1118


In [10]:
# Final Model Saving

print("Saving Final Tuned Pipeline...")

# The grid search best estimator is already refitted on the full dataset (by default refit=True)
final_pipeline = best_pipeline_final

# Ensure target directory exists
os.makedirs('experiments-llm', exist_ok=True)

# Save Pipeline
pipeline_path = 'experiments-llm/best_nn_pipeline.pkl'
joblib.dump(final_pipeline, pipeline_path)
print(f"Pipeline saved to {pipeline_path}")

# Save Label Encoder
le_path = 'experiments-llm/best_nn_label_encoder.pkl'
joblib.dump(le_best, le_path)
print(f"Label Encoder saved to {le_path}")

# Save Metadata
metadata = {
    'embedding': best_embedding,
    'sampling': best_sampling,
    'reduction': best_reduction,
    'best_params': grid_search.best_params_
}
metadata_path = 'experiments-llm/best_nn_metadata.pkl'
joblib.dump(metadata, metadata_path)
print(f"Metadata saved to {metadata_path}.")

Saving Final Tuned Pipeline...
Pipeline saved to experiments-llm/best_nn_pipeline.pkl
Label Encoder saved to experiments-llm/best_nn_label_encoder.pkl
Metadata saved to experiments-llm/best_nn_metadata.pkl.


In [21]:
def build_model(X,y):
    num_classes = len(set(y))

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.4))
    # model.add(layers.Dense(128, activation='relu'))
    # model.add(layers.Dropout(0.3))
    # model.add(layers.Dense(64, activation='relu'))
    # model.add(layers.Dropout(0.3))

    # final layer for multi-class classification
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    # print(model.summary())

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        verbose=0
    )
    loss, acc = model.evaluate(X_val, y_val)
    print("Validation accuracy:", acc)
    return model

build_model(X_best, y_best)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7902 - loss: 0.9017 
Validation accuracy: 0.7902097702026367
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7902 - loss: 0.9017 
Validation accuracy: 0.7902097702026367


<Sequential name=sequential_390, built=True>